# Case definitions: burns and head injuries

**Nicholas Soulakis, PhD** · October 2026

**Summary.** Surveillance starts by deciding what counts as a case. This notebook applies two case definitions, burns and head injuries, to ten years of national emergency department records (NEISS, 2014–2023, about 3.5 million visits) and counts the cases each one finds.

*The method is adapted from Methods in Health and Biomedical Informatics (HSIP 441), taught by the author at Northwestern University Feinberg School of Medicine.*

## Why it matters

A surveillance system can only watch for what it can define. A case definition turns a question ("are burns rising?") into a rule a computer can apply to every record, every day. Get the definition wrong and every signal downstream is wrong too.

In [1]:
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("../utils").exists():
    os.system("git clone -q https://github.com/nsoulakis/public_health_surveillance")
    os.chdir("public_health_surveillance/python/notebooks")
sys.path.insert(0, "..")

In [2]:
from utils.neiss import case_counts, code_cases, read_year, read_years

raw = Path(os.environ.get("NEISS_DIR", "../../data/raw/neiss"))
FULL = any(raw.glob("neiss20*"))
if FULL:
    df = read_years(raw, range(2014, 2024))
else:
    df = read_year(Path("../../data/demo/neiss/neiss_demo.csv"))
print("Mode:", "full data" if FULL else "demo sample (2%)", "|", f"{len(df):,} visits",
      "|", df["trmt_date"].min().date(), "to", df["trmt_date"].max().date())

Mode: full data | 3,520,523 visits | 2014-01-01 to 2023-12-31


## The method

Each visit carries up to two diagnosis codes, each paired with a body part. The definitions:

- **Burn:** a burn diagnosis (electrical, scald, chemical, thermal, or not specified) in either diagnosis field. Missing when both fields are blank.
- **Head injury:** a concussion, fracture, hematoma, laceration or internal injury, *on the head*, in either diagnosis and body-part pair.

Each visit is coded 1 (case) or 0 (not a case). NEISS is a probability sample, so each record also carries a weight; summing the weights of the cases gives a national estimate.

In [3]:
df = code_cases(df)
counts = case_counts(df, ["burn", "head_injury"])
counts

,definition,cases,non_cases,missing,national_estimate
0,burn,60645,3459878,0,2068562
1,head_injury,571520,2949003,0,19920956


In [4]:
by_year = (df.assign(year=df["trmt_date"].dt.year)
             .groupby("year")[["burn", "head_injury"]].sum().astype(int))
by_year

,burn,head_injury
year,,
2014,6200,53314
2015,6255,52896
2016,6296,55632
2017,6444,57568
2018,6090,55989
2019,6051,61572
2020,5963,53174
2021,6037,58839
2022,5601,60705


## Where else it works

The same pattern (a rule over coded fields, applied to every record, with a weight for national estimates) fits any coded surveillance source: syndromic emergency department feeds, poison center calls, reportable disease registries, crime reports. Only the codes change.

## Going further

- Add a third definition, for example falls on stairs (product code plus a fall diagnosis), and compare its counts.
- The narratives often name what the codes miss. Search them for a term and see how many cases the codes alone would have found.
- Next notebook: turning these cases into daily signals.